# 05 — End-to-End Walkthrough

**Purpose:** A guided, user-oriented journey from raw pipeline artefacts through
database exploration, sequence retrieval, and a downstream result snapshot — all in
one cohesive story.

If you are new to PBI, run this notebook first.  It references individual notebooks
for deeper dives on each topic.

| | |
|---|---|
| **Expected inputs** | Pipeline logs at `/pipeline-logs`, database + FASTA files via `pbi.quick_connect()` |
| **Outputs** | Summary tables saved under `<results>/05_end_to_end_walkthrough/` |
| **Companion notebooks** | `00` logs · `01` database QC · `02` retrieval · `03` ML · `06` reproducibility |

## Story arc

1. [Setup & environment](#setup)
2. [Step 1 — Pipeline provenance check](#step1)
3. [Step 2 — Database at a glance](#step2)
4. [Step 3 — Query phage–host pairs](#step3)
5. [Step 4 — Retrieve sequences](#step4)
6. [Step 5 — Downstream analysis snapshot](#step5)
7. [Takeaways & next steps](#takeaways)


---
<a id='setup'></a>
## Setup


In [1]:
import sys
import json
import os
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import pbi
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

# Results directory
results_root = Path(os.getenv('PBI_RESULTS_DIR', '/results'))
results_dir = results_root / '05_end_to_end_walkthrough'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'pbi version  : {pbi.__version__}')
print(f'Python       : {sys.version.split()[0]}')
print(f'Results dir  : {results_dir}')


pbi version  : 0.4.0
Python       : 3.10.20
Results dir  : /results/05_end_to_end_walkthrough


---
<a id='step1'></a>
## Step 1 — Pipeline Provenance Check

Before looking at data, confirm that the pipeline ran and understand *which*
data snapshot it consumed.  The provenance JSON written by the pipeline records
the snapshot date, the provider schema profile, and the build metadata.

See `00_pipeline_logs.ipynb` for a full exploration of all log artefacts.


In [2]:
logs_root = Path('/pipeline-logs')
prov_path = logs_root / 'csv' / 'pipeline_run_provenance.json'

if prov_path.exists():
    provenance = json.loads(prov_path.read_text())
    print('Pipeline run provenance')
    print('─' * 50)
    for key, val in provenance.items():
        print(f'  {key:<40} {val}')
else:
    provenance = {}
    print('ℹ️  pipeline_run_provenance.json not found.')
    print('   Run the Snakemake pipeline first, or mount ./pipeline_logs.')

# Quick sanity check on log directories
print()
for sub in ('logs', 'csv', 'reports'):
    d = logs_root / sub
    status = f'✅  {len(list(d.iterdir()))} files' if d.exists() else '❌  not found'
    print(f'  pipeline-logs/{sub}/  {status}')


Pipeline run provenance
──────────────────────────────────────────────────
  pipeline_run_timestamp                   2026-07-15T06:24:02Z
  provider_name                            PhageScope
  provider_release                         rolling
  provider_snapshot_date                   2026-07-05
  provider_schema_profile                  phagescope_v2
  provider_api_base_url                    https://phageapi.deepomics.org
  provider_provenance_mode                 config_pinned
  pbi_version                              
  git_commit                               
  download_records_count                   229

  pipeline-logs/logs/  ✅  9 files
  pipeline-logs/csv/  ✅  13 files
  pipeline-logs/reports/  ✅  12 files


---
<a id='step2'></a>
## Step 2 — Database at a Glance

`pbi.quick_connect()` auto-detects the optimised DuckDB database and all FASTA
indexes.  We use `get_stats()` for a one-shot overview.

Detailed QC and distribution plots live in `01_database_exploration.ipynb`.


In [3]:
retriever = pbi.quick_connect()
stats = retriever.get_stats()

print('Database overview')
print('─' * 50)
for category, values in stats.items():
    print(f'\n{category.upper()}')
    for key, value in values.items():
        if isinstance(value, (int, float)):
            print(f'  {key:<36} {value:,}')
        else:
            print(f'  {key:<36} {value}')


2026-07-28 06:45:35,775 - INFO - 📂 Checking FASTA index files:
2026-07-28 06:45:35,777 - INFO -    Phage index: True (82826.5 KB)
2026-07-28 06:45:35,779 - INFO -    Protein index: True (4376442.2 KB)
2026-07-28 06:45:35,800 - INFO - 📂 Loaded private phage mapping for 2 sources: ['test_private', 'test_private_2']
2026-07-28 06:45:35,801 - INFO - 📂 Using host mapping file: /data/processed/sequences/host_fasta_mapping.json
2026-07-28 06:45:35,825 - INFO -    Loaded mapping for 5519 hosts
2026-07-28 06:45:35,827 - INFO - 📂 Connecting to database: /data/processed/databases/phage_database_optimized.duckdb
2026-07-28 06:45:35,889 - INFO - 🔄 Starting background FASTA loading...
2026-07-28 06:45:35,891 - INFO - 🔄 [Background] Loading phage FASTA: /data/processed/sequences/all_phages.fasta
2026-07-28 06:45:35,891 - INFO - ✅ Initialization complete (FASTA loading in background)
2026-07-28 06:45:35,894 - INFO - ⏳ Waiting for FASTA loading to complete...
2026-07-28 06:45:48,293 - INFO -    ✅ Phage

Database overview
──────────────────────────────────────────────────

DATABASE
  phages                               1,350,644
  proteins                             71,971,209
  hosts                                5,522
  phage_host_associations              1,240,905
  source_breakdown                     [{'source_type': 'private', 'Source_DB': 'test_private', 'count': 5}, {'source_type': 'private', 'Source_DB': 'test_private_2', 'count': 3}, {'source_type': 'public', 'Source_DB': 'MetaVR', 'count': 225269}, {'source_type': 'public', 'Source_DB': 'GOV2', 'count': 195699}, {'source_type': 'public', 'Source_DB': 'MGV', 'count': 189680}, {'source_type': 'public', 'Source_DB': 'IMGVR', 'count': 177361}, {'source_type': 'public', 'Source_DB': 'GPD', 'count': 142809}, {'source_type': 'public', 'Source_DB': 'TemPhD', 'count': 66823}, {'source_type': 'public', 'Source_DB': 'ELGV', 'count': 56716}, {'source_type': 'public', 'Source_DB': 'CHVD', 'count': 44935}, {'source_type': 'public', 'S

---
<a id='step3'></a>
## Step 3 — Query Phage–Host Pairs

We retrieve a small, tractable set of phage–host pairs to work with
through the rest of this walkthrough.  The query filters to *lytic* phages
linked to *Escherichia coli* hosts that have a downloadable assembly — a
biologically meaningful and data-complete subset.

Adjust the filter to suit your research question.


In [10]:
pairs_df = retriever.query_phage_host_pairs(
    phage_filters={'Lifestyle': 'virulent'},
    limit=20,
)
pairs_df

2026-07-28 08:08:36,388 - INFO - 🔍 Querying phage-host pairs...
2026-07-28 08:08:37,325 - INFO - 📊 Found 20 phage-host pairs
2026-07-28 08:08:37,327 - INFO - 📥 Fetching sequences for 20 phages and 10 unique hosts
2026-07-28 08:08:38,196 - INFO - ✅ Retrieved 20 complete phage-host pairs with sequences


,Phage_ID,Host_ID,Phage_Source,Phage_Source_Type,Phage_Length,Phage_GC,Phage_Taxonomy,Phage_Completeness,Phage_Lifestyle,Phage_Cluster,Phage_Subcluster,Species_Name,Host_Assembly_Level,Host_Length,Host_GC,Host_RefSeq_Category,Phage_Sequence,Host_Sequence
0,MGV-GENOME-0344764,GCF_050858935_1,MGV,public,58473,46.665983,Caudovirales,Medium-quality,virulent,cluster_159326,subcluster_200614,Anaerostipes hadrus,Complete Genome,3071641,37.05,na,CTCCCCTCGCTACTCGGAAGCAAGCTTCCTCGTACAAATACATGTTGACAGATAAC...,TGTTGAAGATTTAGTTCAGATAATAACTGTATTTAGTTGTAAATTACAAGGCAAAC...
1,MGV-GENOME-0344789,GCF_976985875_1,MGV,public,66107,53.026155,Caudovirales,Medium-quality,virulent,cluster_77087,subcluster_96992,Phocaeicola vulgatus,Complete Genome,5230292,42.59,na,ATATATTTCTTGTATCGTATTATCGCGCTTACCGTCTGGCGGCTGACGCCGTACTG...,GAGAGACAGGCTCTCGAACCTATACATTCAAGAAATATCATAAAATTGCAAATTAC...
2,MGV-GENOME-0344929,GCF_055398035_1,MGV,public,41720,42.272291,Caudovirales,Medium-quality,virulent,cluster_255519,subcluster_320840,Bacteroides reticulotermitis,Complete Genome,5470359,43.26,reference genome,AAATGAGCCTTGGGCGGGCTTTGTAAAACCCACATTAAAAAATATGGATAAAATTA...,TGCGATTCATCAACAAATAAATGGTCGATGCCCATCAGCCTGAAATCCACCGCATC...
3,MGV-GENOME-0345030,GCF_050858935_1,MGV,public,69883,36.956055,Caudovirales,Medium-quality,virulent,cluster_113664,subcluster_143308,Anaerostipes hadrus,Complete Genome,3071641,37.05,na,GCTTGGAATAAGGCAAGTAAAAAAGAACACAAAAAACTTCTTGACCTGCCTAACTG...,TGTTGAAGATTTAGTTCAGATAATAACTGTATTTAGTTGTAAATTACAAGGCAAAC...
4,MGV-GENOME-0345236,GCF_976985775_1,MGV,public,68280,42.362332,Caudovirales,High-quality,virulent,cluster_176243,subcluster_222435,Parabacteroides merdae,Complete Genome,4388850,45.32,na,TCTGTTCTTTTGACATTTAGATAGGGAAAAGAGGGTTGAACATGCTTGTGTATTAT...,CTTGAATAGTCGTTTGGCCAGTCATCAACATCACCAGACGGTCGATGCCGATACCG...
5,MGV-GENOME-0345248,GCF_058182505_1,MGV,public,56234,51.077640,Caudovirales,High-quality,virulent,cluster_99464,subcluster_125336,Alistipes putredinis,Complete Genome,2350222,53.73,na,CGACTATAGCAAATAATAATGATAAAAAAGTCCTTAATGTTACTGTTTTACAATCG...,ATGTTAGCTCCCGCACAGACATATTGCGAAAAATGGCAAACTTGTTTGGACCAGAT...
6,MGV-GENOME-0345278,GCF_051201525_1,MGV,public,62549,51.243025,Caudovirales,High-quality,virulent,cluster_256640,subcluster_322227,Faecalibacterium prausnitzii,Complete Genome,3135029,57.13,na,TACGAAGCATCGAATAAAATGCGAATCCATGATGTATTTGAACGTGATTGAATGCC...,AGCTCCTGTTCCAAGCCGTTTCAGCTAACTATCTTGGGCACGAGTGAGCTTAACTG...
7,MGV-GENOME-0345326,GCF_055398035_1,MGV,public,63493,37.843542,Caudovirales,High-quality,virulent,cluster_125762,subcluster_158361,Bacteroides reticulotermitis,Complete Genome,5470359,43.26,reference genome,TGCCGGTTCTATTGTTTAACTTTTAATTTTCCGGTAAAAATTTACCGAAGTATTGT...,TGCGATTCATCAACAAATAAATGGTCGATGCCCATCAGCCTGAAATCCACCGCATC...
8,MGV-GENOME-0340511,GCF_976985875_1,MGV,public,44853,40.855684,Caudovirales,Medium-quality,virulent,cluster_147702,subcluster_186282,Phocaeicola vulgatus,Complete Genome,5230292,42.59,na,TAAATCAGAGATTAATTTCATAGTGATAGACTTTGAAACGGCTACCTCCAAACGTG...,GAGAGACAGGCTCTCGAACCTATACATTCAAGAAATATCATAAAATTGCAAATTAC...
9,MGV-GENOME-0340775,GCF_976985875_1,MGV,public,61708,52.955857,Caudovirales,Medium-quality,virulent,cluster_277453,subcluster_348565,Phocaeicola vulgatus,Complete Genome,5230292,42.59,na,AAAAGTATCGTTTTCCAATTACTACAAATAACGGATTTGTATTTGCGAGGTCATTC...,GAGAGACAGGCTCTCGAACCTATACATTCAAGAAATATCATAAAATTGCAAATTAC...


In [15]:
pairs_df = retriever.query_phage_host_pairs(
    phage_filters={'Lifestyle': 'virulent'},
    host_filters={'Organism_Name': '%Escherichia%'},
    limit=20,
)

print(f'Pairs returned: {len(pairs_df)}')

if not pairs_df.empty:
    # Show a tidy subset of columns if they exist
    preview_cols = [c for c in [
        'Phage_ID', 'Phage_Name', 'Lifestyle', 'Genome_Length',
        'GC_Content', 'Host_Assembly_Accession', 'Organism_Name'
    ] if c in pairs_df.columns]
    display(pairs_df[preview_cols].head(10))
else:
    print('No pairs found — try relaxing the filter or check the database.')


2026-07-28 08:08:53,787 - INFO - 🔍 Querying phage-host pairs...
2026-07-28 08:08:54,068 - INFO - 📊 Found 20 phage-host pairs
2026-07-28 08:08:54,070 - INFO - 📥 Fetching sequences for 20 phages and 2 unique hosts
2026-07-28 08:08:54,707 - INFO - ✅ Retrieved 20 complete phage-host pairs with sequences


Pairs returned: 20


,Phage_ID
0,MGV-GENOME-0341174
1,MGV-GENOME-0312826
2,MGV-GENOME-0222840
3,IMGVR_UViG_3300045988_148266|3300045988|Ga0495776_080053
4,IMGVR_UViG_3300045988_007060|3300045988|Ga0495776_139485
5,IMGVR_UViG_3300045988_007740|3300045988|Ga0495776_153710
6,IMGVR_UViG_3300048825_000055|3300048825|Ga0495405_000124
7,Station205_SUR_ALL_assembly_NODE_3200_length_14265_cov_1...
8,Station194_SUR_ALL_assembly_NODE_1159_length_28139_cov_1...
9,Station188_DCM_ALL_assembly_NODE_6796_length_12187_cov_3...


In [16]:
# Save the pairs table for later reference
if not pairs_df.empty:
    out_path = results_dir / 'lytic_ecoli_pairs_sample.csv'
    pairs_df.to_csv(out_path, index=False)
    print(f'Saved {len(pairs_df)} rows → {out_path}')


Saved 20 rows → /results/05_end_to_end_walkthrough/lytic_ecoli_pairs_sample.csv


---
<a id='step4'></a>
## Step 4 — Retrieve Sequences

For the first few pairs we now pull the actual DNA sequences.  Because FASTA
files are pyfaidx-indexed, each retrieval is a direct byte-offset read — there
is no need to scan the whole file.

> **Note:** Some pairs may not have a host genome downloaded (host assembly exists
> in the database but the FASTA was not fetched).  These are silently skipped.
> See `00_pipeline_logs.ipynb § 4` for how to inspect download failures.


In [17]:
MAX_PAIRS = 5  # Keep this small for the walkthrough

retrieved = []
if not pairs_df.empty:
    phage_ids = pairs_df['Phage_ID'].dropna().unique().tolist()
    for phage_id in phage_ids[:MAX_PAIRS]:
        try:
            seq = retriever.get_phage_sequence(phage_id)
            if seq:
                retrieved.append({'Phage_ID': phage_id, 'Seq_length': len(seq), 'GC': round(
                    (seq.count('G') + seq.count('C')) / len(seq) * 100, 2
                ) if seq else None})
                print(f'  {phage_id:<30}  {len(seq):>8,} bp')
        except Exception as exc:
            print(f'  {phage_id:<30}  ⚠️  {exc}')

if not retrieved:
    print('No sequences retrieved — FASTA files may not be available in this environment.')


  MGV-GENOME-0341174                55,896 bp
  MGV-GENOME-0312826                48,624 bp
  MGV-GENOME-0222840                31,645 bp
  IMGVR_UViG_3300045988_148266|3300045988|Ga0495776_080053    17,510 bp
  IMGVR_UViG_3300045988_007060|3300045988|Ga0495776_139485    43,620 bp


---
<a id='step5'></a>
## Step 5 — Downstream Analysis Snapshot

Even a small retrieval is enough to compute simple bioinformatic summaries:
genome length distribution and GC content, two of the most basic phage
characterisation metrics.  The `03_ml_streaming.ipynb` notebook builds on
these features for a full ML workflow.


In [18]:
# ── Genome length & GC distribution from the pair query ─────────────────
if not pairs_df.empty:
    cols_to_plot = [c for c in ['Genome_Length', 'GC_Content'] if c in pairs_df.columns]
    if cols_to_plot:
        fig, axes = plt.subplots(1, len(cols_to_plot), figsize=(6 * len(cols_to_plot), 4))
        if len(cols_to_plot) == 1:
            axes = [axes]
        for ax, col in zip(axes, cols_to_plot):
            pairs_df[col].dropna().hist(ax=ax, bins=20, edgecolor='k', alpha=0.7)
            ax.set_title(col.replace('_', ' '))
            ax.set_xlabel(col)
            ax.set_ylabel('Count')
        plt.suptitle('Lytic E. coli-infecting phages — sample distribution', y=1.02)
        plt.tight_layout()
        fig_path = results_dir / 'sample_distributions.png'
        plt.savefig(fig_path, dpi=96, bbox_inches='tight')
        plt.show()
        print(f'Figure saved → {fig_path}')
    else:
        print('Genome_Length / GC_Content columns not present in this query result.')
else:
    print('No data to plot.')


Genome_Length / GC_Content columns not present in this query result.


---
<a id='takeaways'></a>
## Takeaways & Next Steps

This walkthrough demonstrated the full PBI data path in five steps:

| Step | What happened |
|------|---------------|
| 1 | Verified the pipeline ran and identified the data snapshot |
| 2 | Got a one-screen database summary (counts, coverage) |
| 3 | Queried a biologically meaningful phage–host subset |
| 4 | Retrieved phage DNA sequences with O(1) FASTA access |
| 5 | Computed basic genome statistics as a downstream result |

### Where to go next

| Goal | Notebook |
|------|----------|
| Inspect all pipeline logs in depth | `00_pipeline_logs.ipynb` |
| Full database QC and distribution plots | `01_database_exploration.ipynb` |
| Exhaustive sequence retrieval API | `02_sequence_retrieval.ipynb` |
| ML feature engineering and training | `03_ml_streaming.ipynb` |
| Release consumer / Zenodo download | `04_data_release_exploration.ipynb` |
| Reproducibility and provenance deep-dive | `06_reproducibility.ipynb` |


In [9]:
# Clean up
#retriever.close()
#print('Database connection closed.')
